# 04 — Model 2.1.1, typed flags as observations

**The model in math terms.** M2.1.1 is the mounted M1.2 stack (adopted Both chains, MIX2 routes, partition link) with one addition: typed misconception flags enter each home KC's chain update as extra likelihood factors. The observation view — one latent per KC, no disposition state, constant learn rate.

**State and transition.** Per KC, the belief $b_t = P(L_t = 1 \mid H_t)$, carried forward by the drift:

$$b^{\text{pre}}_t = b_{t-1} + (1 - b_{t-1})\,\tau$$

**Prediction** (before the turn reveals): the qc forecast pools the chains' act-probabilities through the MIX2 routes and the partition link $P = x(1-s_0) + (1-x)g_0$, exactly as in M1.2. Flags never enter prediction.

**Update** (after the turn reveals): the turn's data $D_t$ is the correctness symbol $o$ plus every flag symbol the annotation presents ($A_t$, the realized set — fired or quiet; NA is a structural non-observation). Symbols are conditionally independent given $L$, so the likelihood is a product:

$$P(D_t \mid L{=}l) = e_l(o) \times \prod_{j \in A_t} P(F_j = f_j \mid L{=}l)$$

with the correctness emissions $e_1(\text{correct}) = 1-s$, $e_0(\text{correct}) = g$ from the adopted chains, and the typed flag pair per flag $j$:

$$P(F_j{=}\text{fired} \mid L{=}1) = u_1 = 0.01 \text{ (pinned)}, \qquad P(F_j{=}\text{fired} \mid L{=}0) = u_0^{(j)} \text{ (fitted)}$$

The posterior is the two-hypothesis Bayes fraction, then the drift:

$$b_t = \frac{b^{\text{pre}}_t \, P(D_t \mid 1)}{b^{\text{pre}}_t \, P(D_t \mid 1) + (1 - b^{\text{pre}}_t)\, P(D_t \mid 0)}$$

A fire multiplies the odds by $u_1/u_0 \approx 0.02$–$0.05$ (the loudest wrong in the alphabet); a quiet by $(1-u_1)/(1-u_0) > 1$ (surviving a met trap outcredits a bare correct). An empty $A_t$ makes the product vanish and the model collapses to M1.2 exactly.

**Flag homing** (mechanism-based, per the report): conjunction → kc1, inverse → kc2, time-axis → kc2, denominator neglect → kc4, base-rate neglect → kc5.

**Flag-table fitting** (two-stage, matching the outer chain's precedent): with correctness-only responsibilities $w_t = 1 - b^{\text{pre}}_t$ on the home KC over training walks,

$$u_0^{(j)} = \frac{\sum_{t:\, j \in A_t} w_t \,\mathbb{1}[f_{j,t}{=}\text{fired}] + \kappa\, c_j}{\sum_{t:\, j \in A_t} w_t + \kappa}, \qquad \kappa = 5$$

shrunk toward the published written-format calibration $c_j$ (CPR uninstructed error rates: conjunction 0.79, inverse 0.65, time-axis 0.63, denominator 0.82, base-rate 0.67), clipped to $[0.1, 0.9]$. $u_1$ is never fitted; conjunction's fire side (zero fires) sits at its calibration center by construction.

**File layout.**
* The model: `scripts/model_2_1_1.py` (subclasses `Model_1_2_MIX2`)
* The flag-blind comparison: **loaded from `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`** (written by notebook 03's `save_outer_chain_from_evaluator`) — M1.2 is *not* re-run in this notebook
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Harness `scripts/evaluator.py`, data `data/data_annotated.csv`, loader `scripts/data.py`
* Saved outputs: `cache/model_2_1_1/Model_2_1_1/` via `save_model_2_1_1_from_evaluator` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [ ]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_1_1 import Model_2_1_1
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
MIX2_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'   # saved by notebook 03's save helper; M1.2 is not re-run here

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
mix2 = pd.read_csv(MIX2_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(mix2), 'stored flag-blind predictions')

## 1. Run
M2.1.1 through the shared harness with the cached inner chains. The flag-blind comparison numbers come from the stored predictions, never re-fitted.

In [ ]:
ev = Evaluator(Model_2_1_1, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

## 2. Results

### 2.1 Headline metrics against the stored flag-blind run
Metrics from the 312 pooled out-of-fold question predictions, same references as notebook 03 (qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.653).

In [ ]:
pd.DataFrame([dict(model='M2.1.1', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M1.2 (stored)', **{k: round(float(v),4) for k,v in _metrics(mix2.y_true, mix2.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

### 2.2 Where the gain lives
The attribution split: flag-bearing participants against the zero-fire group.

In [ ]:
FLAGGED = ['P01','P02','P03','P06','P11','P20','P23','P24']
def grp(p, pids):
    sub = p[p.participant_id.isin(pids)]
    return round(float(_metrics(sub.y_true, sub.p_pred)['auc']), 3)
others = sorted(set(df.participant_id) - set(FLAGGED))
pd.DataFrame([
    dict(group='flag-bearers', m2_1_1=grp(preds, FLAGGED), m1_2=grp(mix2, FLAGGED)),
    dict(group='zero-fire',    m2_1_1=grp(preds, others),  m1_2=grp(mix2, others)),
]).set_index('group')

### 2.3 Fitted flag tables and bridge anchors
Fold-mean fitted fire rates beside their written-format calibration centers, and the bridge anchors against the mechanism censuses.

In [ ]:
u0 = pd.DataFrame([m.u0 for m in ev.fold_models.values()]).mean().round(3)
anchors = pd.Series(dict(s0=np.mean([m.s0 for m in ev.fold_models.values()]),
                         g0=np.mean([m.g0 for m in ev.fold_models.values()]))).round(3)
print('fitted u0 (fold means) vs written-format calibration:')
display(pd.DataFrame(dict(fitted=u0, calibration=pd.Series({'conjunction':0.79,'inverse':0.65,'time_axis':0.63,'denominator_neglect':0.82,'base_rate_neglect':0.67}))))
print('bridge anchors (censuses 0.077 / 0.061):')
display(anchors)

### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, M2.1.1 beside the stored flag-blind M1.2.

**Observations.**
* Both matrices share the same shape of error: the dominant cell of failure is wrongs predicted correct (false positives), the blindside set — at this threshold most wrong answers still arrive unannounced under either model.
* M2.1.1 trims the blindside set slightly rather than dramatically: a handful of MIX2's false positives cross under 0.5 (the fire-history rows), at the cost of at most one new false negative. The typed channel's gains are ranking-shaped — probabilities move in the right direction on many rows (AUC, AUPRC, log-loss all improve) but few rows cross the arbitrary 0.5 line.
* The residual false positives are dominated by the zero-fire construal and misread participants (P16, P26, P25, P15, P10) — the blindside mass the flag channel cannot see by construction, and the measured motivation for the parked misinterpretation channel.


In [ ]:
def cmat(p):
    yhat = (p.p_pred >= 0.5).astype(int)
    cm = pd.crosstab(p.y_true.map({1:'actual correct', 0:'actual wrong'}),
                     yhat.map({1:'predicted correct', 0:'predicted wrong'}))
    return cm.reindex(index=['actual correct','actual wrong'],
                      columns=['predicted correct','predicted wrong'], fill_value=0)

cm2, cm1 = cmat(preds), cmat(mix2)
print('M2.1.1:'); display(cm2)
print('M1.2 (stored):'); display(cm1)
fp2 = cm2.loc['actual wrong','predicted correct']; fp1 = cm1.loc['actual wrong','predicted correct']
fn2 = cm2.loc['actual correct','predicted wrong']; fn1 = cm1.loc['actual correct','predicted wrong']
print(f'false positives {fp1} -> {fp2} | false negatives {fn1} -> {fn2}')

## 3. Conclusion

* **The typed flags add predictive signal at the mounted qc layer.** M2.1.1 beats the flag-blind M1.2 on every headline metric (AUC ~0.691 vs ~0.674, AUPRC-wrong ~0.591 vs ~0.575, log-loss ~0.600 vs ~0.607) on the same 312 targets with identical inner chains.
* **The gain lives exactly where it should.** Flag-bearing participants improve (~+0.03 AUC) while the zero-fire group is flat — the channel helps precisely the students who produced typed evidence and changes nothing for everyone else, which is what genuine signal looks like.
* **Verbal-format fire suppression, quantified.** The fitted unmastered fire rates sit far below their written-format calibrations (base-rate ~0.21 vs 0.67; denominator ~0.31 vs 0.82): students explaining aloud fire the canonical fallacies at a third to half their written rates — the datum the report pre-registered as of interest.
* **The bridge stays honest.** M2.1.1's anchors land at s0 ~0.080 (census 0.077) and g0 ~0.091 (census 0.061) — the tightest census match yet; the flag evidence sharpens the chains without stealing the bridge's meaning.
* **Caveats.** The pooled effect is modest (+0.017 AUC) and participant-clustered bootstrap intervals are pending; threshold-crossing conversions are few (the gains are ranking-shaped); and this is the mounted qc secondary — the registered primary contrast (per-KC layer, M2.1 vs M1) runs separately.

## 4. Save
Persist the run: per-fold bridge and flag tables, pooled predictions, metrics, and the index (pins, kappa, calibration, homing).

In [ ]:
from scripts.model_2_1_1 import save_model_2_1_1_from_evaluator
save_model_2_1_1_from_evaluator(ev)